# Text classification with SVMs

In this notebook, we will use SVMs to find who wrote a given text among several possibilities.

We will use a corpus of classical English literature: 9 authors, 2 books per author, a file per chapter.

Let's start by getting the data:

In [ ]:
spacy_updated = !pip freeze | grep spacy==3
if not spacy_updated:
  !pip install 'spacy >= 3, < 4'

In [ ]:
! git clone https://github.com/nzmonzmp/dataset-9classical-author.git

In [ ]:
import pathlib
import re
import time
import typing

import matplotlib.pyplot as plt
import numpy
import seaborn
import sklearn.feature_extraction.text
import sklearn.preprocessing
import sklearn.model_selection
import sklearn.svm
import spacy
import tqdm.notebook

## Train set creation

After a quick look at the data, implement the `get_data` function:
- it should take the path of the corpus directory as argument
- it should return two lists:
  - the first should contain the texts
  - the second should contain the corresponding authors

In [ ]:
def get_data(directory: pathlib.Path
             ) -> typing.Tuple[typing.List[str], typing.List[str]]:
  texts = []
  authors = []

  # Your code here

  return texts, authors


texts, authors = get_data(pathlib.Path("dataset-9classical-author"))

### Solution

In [ ]:
def get_data(directory: pathlib.Path
             ) -> typing.Tuple[typing.List[str], typing.List[str]]:
  texts = []
  authors = []
  for item in tqdm.notebook.tqdm(list(directory.glob("*/*/*.txt"))):
    author = item.parent.parent.name
    # Or with parts
    # author = item.parts[1]

    with item.open(encoding="utf8") as fh:
      text = fh.read()
    # Or with read_text
    # text = item.read_text(encoding="utf8")

    texts.append(text)
    authors.append(author)

  return texts, authors


texts, authors = get_data(pathlib.Path("dataset-9classical-author"))

## TF-IDF representation

Use the `TfidfVectorizer` of the `sklearn` library to transform `texts` so that it can be manipulated by a SVM. Store the result in a variable called `X`.

In [ ]:
# Your code here

## Solution

In [ ]:
vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
X = vectorizer.fit_transform(texts)

## Label Encoding

Use the `LabelEncoder` class of `sklearn` to encode `authors`. Store the result in a variable named `y`.

In [ ]:
# Your code here

### Solution

In [ ]:
# With only basic Python
index_to_author = list(set(authors))
author_to_index = {author: index
                   for index, author
                   in enumerate(index_to_author)}
y = numpy.array([author_to_index[author] for author in authors])

# With scikit-learn
le = sklearn.preprocessing.LabelEncoder()
y = le.fit_transform(authors)

## Training and evaluation

Use the `sklearn`'s `cross_val_score` function to evaluate a SVM model.

- Use `sklearn.svm.SVC` with a linear kernel
- Use 5 fold cross-validation
- Use the `time` module (and its `time` function) from the standard library to display the elapsed time to train the 5 folds

In [ ]:
# Your code here

### Solution

In [ ]:
start_time = time.time()
clf = sklearn.svm.SVC(kernel='linear')
scores = sklearn.model_selection.cross_val_score(clf, X, y, cv=5)
print(f"The training took {time.time() - start_time:.2f} seconds")
print(scores)

## Optimization when using a linear kernel

Reuse the code you've just written but use `sklearn.svm.LinearSVC` as model. It is *way* faster when using SVM with a linear kernel. 

In [ ]:
# Your code here

### Solution

In [ ]:
start_time = time.time()
clf = sklearn.svm.LinearSVC()
scores = sklearn.model_selection.cross_val_score(clf, X, y, cv=5)
print(f"The training took {time.time() - start_time:.2f} seconds")
print(scores)

## Conclusions

What can you say about the obtained scores?

### Solution

The performance close to 1 is highly suspicious. We could for example speculate that:

- the size of the texts (chapters) is too important and the model can easily spot at least a few words that are specific to the author.
- the named entities are specific to each work and will appear in all chapters.

We can design ways to test those assumptions:

- train a model at the sentence level to measure the impact of text size
- train a model without named entities to measure the impact of named entities

## Replacing named entities by special markers

Let's use `spacy` to replace named entities by special markers. We'll deactivate everything but the parts of spacy we need.

In [ ]:
#!python -m spacy download en_core_web_lg
!python -m spacy download en_core_web_sm

In [ ]:
nlp = spacy.load(
    "en_core_web_sm",
    exclude=("tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"))
nlp.enable_pipe("senter")

Let's now define the function to replace named entities by special markers:

In [ ]:
doc = nlp("Jimmy Hendrix was absolutely amazing at Woodstock!")

# Display
print(" ".join(f"{t}/{t.ent_type_}/{t.ent_iob_}" if t.ent_type_ else t.text
               for t in doc))

E.g., if the above cell outputs the following:

    Jimmy/PERSON/B Hendrix/PERSON/I was absolutely amazing at Woodstock/ORG/B !

We want our function to output:

    PPPERSON was absolutely amazing at OOORG !

To achieve that, use the [attributes `ent_type_` & `ent_iob_`](https://spacy.io/api/token#attributes) of tokens obtained with the `spacy` model (we triple the first letter to avoid known words).

In [ ]:
def replace_ners(tokens: typing.Iterable[spacy.tokens.Token]) -> str:
  # Your code here
  return ""


replace_ners(doc)

### Solution

In [ ]:
def replace_ners(tokens: typing.Iterable[spacy.tokens.Token]) -> str:
  result = []
  for token in tokens:
    if token.ent_iob_ == "O":
      result.append(token.text)
    elif token.ent_iob_ == "B":
      result.append(f"{token.ent_type_[0] * 2}{token.ent_type_}")
  return " ".join(result)


print(replace_ners(doc))

## Extended `get_data` function

We extend the `get_data` function so that it returns all combinations of chapter level or sentence level, and with or without named entities.

It returns an object that has 6 attributes, that are the 4 versions of `texts`, and the 2 versions of `authors`.

In [ ]:
class Data(typing.NamedTuple):
  texts_sentence_replaced_ners: typing.List[str] 
  texts_sentence_normal: typing.List[str] 
  texts_document_replaced_ners: typing.List[str] 
  texts_document_normal: typing.List[str] 
  authors_sentence: typing.List[str]
  authors_document: typing.List[str]


def get_data(directory: pathlib.Path,
             ) -> typing.Tuple[typing.List[str], typing.List[str]]:
  texts = []
  authors_sentence = []
  authors_document = []
  texts_sentence_replaced_ners = []
  texts_sentence_normal = []
  texts_document_replaced_ners = []
  texts_document_normal = []
  paths = list(directory.glob("*/*/*.txt"))
  contents = (p.read_text(encoding="utf8").replace("\n", " ") for p in paths)
  for path, doc in tqdm.notebook.tqdm(zip(paths, nlp.pipe(contents,
                                                          n_process=-1)),
                                      total=len(paths)):

    author = path.parent.parent.name
    for sentence in doc.sents:
      authors_sentence.append(author)
      texts_sentence_replaced_ners.append(replace_ners(sentence))
      texts_sentence_normal.append(" ".join(token.text for token in sentence))
    authors_document.append(author)
    texts_document_replaced_ners.append(replace_ners(doc))
    texts_document_normal.append(" ".join(token.text for token in doc))
  return Data(authors_sentence=authors_sentence,
              authors_document=authors_document,
              texts_sentence_replaced_ners=texts_sentence_replaced_ners,
              texts_sentence_normal=texts_sentence_normal,
              texts_document_replaced_ners=texts_document_replaced_ners,
              texts_document_normal=texts_document_normal)


data = get_data(pathlib.Path("dataset-9classical-author"))

## Training and evaluation

What is now the performance of a 5-fold cross-validated SVM for each given configuration?

Hint: create a function that applies all the required preprocessing and trains the model, then returns the mean of the scores obtained through `cross_val_score`.

In [ ]:
# Your code here

### Solution

In [ ]:
def evaluate(texts: typing.List[str], authors: typing.List[str]) -> float:
  vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
  X = vectorizer.fit_transform(texts)
  
  le = sklearn.preprocessing.LabelEncoder()
  y = le.fit_transform(authors)
  
  clf = sklearn.svm.LinearSVC()
  scores = sklearn.model_selection.cross_val_score(clf, X, y, cv=5)
  return scores.mean()


print("Document level, with named entities:",
      evaluate(data.texts_document_normal, data.authors_document))
print("Document level, without named entities:",
      evaluate(data.texts_document_replaced_ners, data.authors_document))
print("Sentence level, with named entities:",
      evaluate(data.texts_sentence_normal, data.authors_sentence))
print("Sentence level, without named entities:",
      evaluate(data.texts_sentence_replaced_ners, data.authors_sentence))

## Confusion matrix on test data

Pick a configuration (text size and named entities handling), then:

- Obtain `X` and `y`
- Split `X` and `y` into train and test sets with `train_test_split` from `sklearn`
- Train a model on the train set
- Evaluate the performances of the model on the test set
- Display the confusion matrix (`confusion_matrix` from `sklearn`). You can use `heatmap` from `seaborn` for a nicer output

In [ ]:
# Your code here

### Solution

In [ ]:
vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
X = vectorizer.fit_transform(data.texts_sentence_replaced_ners)

le = sklearn.preprocessing.LabelEncoder()
y = le.fit_transform(data.authors_sentence)

In [ ]:
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, test_size=0.3, random_state=42)

In [ ]:
clf = sklearn.svm.LinearSVC()
clf.fit(X_train, y_train)
clf.score(X_test, y_test)

In [ ]:
y_test_pred = clf.predict(X_test)
conf_mat = sklearn.metrics.confusion_matrix(y_test, y_test_pred)
total = conf_mat.sum(axis=0)
with numpy.printoptions(precision=2, suppress=True):
  print(conf_mat / total)
print(total)

In [ ]:
seaborn.heatmap(conf_mat,
                cmap="rocket_r",
                xticklabels=le.classes_,
                yticklabels=le.classes_,
                annot=True,
                fmt="d")
plt.show()